<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/blind_set_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 52.3 MB/s eta 0:00:00


In [6]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

# 1. Load the blind set
csv_path = "blindset.csv"
df = pd.read_csv(csv_path)

# 2. Setup the SDWriter for the unbundled conformers
writer = Chem.SDWriter("BlindSet_3D.sdf")
successful_mols = 0

for index, row in df.iterrows():
    smiles = row['Smiles']
    mol_id = str(row['ID'])

    # Generate 2D graph
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Failed to parse SMILES for ID {mol_id}")
        continue

    # Add explicit hydrogens
    mol = Chem.AddHs(mol)

    # ETKDGv3 Embedding Setup
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.enforceChirality = True

    # Generate 5 conformers per unique molecule
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=5, params=params)

    # Force Field Optimization and Spatial Isolation
    for cid in conf_ids:
        # MMFF94 Optimization with 500 max iterations
        try:
            opt_status = AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=500)

            # Catch silent RDKit forcefield failures
            if opt_status == -1:
                print(f"Missing FF parameters for ID {mol_id}, Conf {cid}. Skipping.")
                continue
            elif opt_status == 1:
                print(f"ID {mol_id}, Conf {cid} did not converge in 500 iterations.")

        except Exception as e:
            print(f"Hard crash during optimization for ID {mol_id}, Conf {cid}: {e}")
            continue

        # Unbundling algorithm: isolate each conformer into a clean object
        single_conf_mol = Chem.Mol(mol)
        single_conf_mol.RemoveAllConformers()

        # Create a detached copy of the conformer
        conf_copy = Chem.Conformer(mol.GetConformer(cid))
        conf_copy.SetId(0) # Force index to 0 for safe SDWriter export

        single_conf_mol.AddConformer(conf_copy, assignId=False)

        # Assign unique, distinct names to each conformer
        single_conf_mol.SetProp("_Name", f"Compound_{mol_id}_conf_{cid}")
        single_conf_mol.SetProp("Parent_ID", mol_id)

        # Save to SDF
        writer.write(single_conf_mol)
        successful_mols += 1

writer.close()
print(f"3D Embedding and Optimization Complete. {successful_mols} geometries saved to BlindSet_3D.sdf")

3D Embedding and Optimization Complete. 320 geometries saved to BlindSet_3D.sdf
